## bills data eda

In [68]:
# imports
import pandas as pd
import re
import seaborn as sns
sns.set()


In [69]:
df = pd.read_csv(
    "../data/bills/bills1.csv",
    skiprows=2
)

In [70]:
df.head()

,Legislation Number,URL,Congress,Title,Sponsor,Date of Introduction,Committees,Latest Action,Latest Action Date,Number of Cosponsors,Amends Bill,Date Offered,Date Submitted,Date Proposed,Amends Amendment
0,H.R. 7933,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),"To amend title 10, United States Code, to expa...","Watson Coleman, Bonnie [Rep.-D-NJ-12]",03/12/2026,House - Armed Services,Referred to the House Committee on Armed Servi...,03/12/2026,0,NaN,NaN,NaN,NaN,NaN
1,H.R. 7932,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),HONOR Gold Star Families Act,"Van Epps, Matt [Rep.-R-TN-7]",03/12/2026,House - Armed Services,Referred to the House Committee on Armed Servi...,03/12/2026,11,NaN,NaN,NaN,NaN,NaN
2,H.R. 7931,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),To amend the Employee Retirement Income Securi...,"Van Drew, Jefferson [Rep.-R-NJ-2]",03/12/2026,House - Education and Workforce,Referred to the House Committee on Education a...,03/12/2026,1,NaN,NaN,NaN,NaN,NaN
3,H.R. 7930,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),"To amend title 18, United States Code, to incr...","Tlaib, Rashida [Rep.-D-MI-12]",03/12/2026,House - Judiciary,Referred to the House Committee on the Judiciary.,03/12/2026,6,NaN,NaN,NaN,NaN,NaN
4,H.R. 7929,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),To direct the Secretary of Transportation to i...,"Titus, Dina [Rep.-D-NV-1]",03/12/2026,"House - Science, Space, and Technology","Referred to the House Committee on Science, Sp...",03/12/2026,0,NaN,NaN,NaN,NaN,NaN


In [71]:
# investigating csv
print(f'df.shape: {df.shape}')
print(f'df.columns: {df.columns}')
print(f'df.info(): {df.info()}')
print(f'df.describe(include="all"): {df.describe(include="all")}')

df.shape: (2500, 15)
df.columns: Index(['Legislation Number', 'URL', 'Congress', 'Title', 'Sponsor',
       'Date of Introduction', 'Committees', 'Latest Action',
       'Latest Action Date', 'Number of Cosponsors', 'Amends Bill',
       'Date Offered', 'Date Submitted', 'Date Proposed', 'Amends Amendment'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Legislation Number    2500 non-null   object 
 1   URL                   2500 non-null   object 
 2   Congress              2500 non-null   object 
 3   Title                 2500 non-null   object 
 4   Sponsor               2500 non-null   object 
 5   Date of Introduction  2500 non-null   object 
 6   Committees            2500 non-null   object 
 7   Latest Action         2500 non-null   object 
 8   Latest Action Date    2500 non-null   object 


In [72]:
# convert to datetime format
date_cols = [
    "Date of Introduction",
    "Latest Action Date",
    "Date Offered",
    "Date Submitted",
    "Date Proposed"
]

In [73]:
# date range (work in progress, need to get more data)
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print(f'min date: {df["Date of Introduction"].min()}')
print(f'max date: {df["Date of Introduction"].max()}')

min date: 2025-09-17 00:00:00
max date: 2026-03-12 00:00:00


In [74]:
# missing vals
missing = df.isnull().sum().sort_values(ascending=False)
print(missing)

Amends Bill             2500
Date Offered            2500
Date Submitted          2500
Date Proposed           2500
Amends Amendment        2500
Legislation Number         0
URL                        0
Congress                   0
Title                      0
Sponsor                    0
Date of Introduction       0
Committees                 0
Latest Action              0
Latest Action Date         0
Number of Cosponsors       0
dtype: int64


In [75]:
# List of columns to drop
drop_cols = [
    "Date Offered",
    "Date Submitted",
    "Date Proposed",
    "Amends Bill",
    "Amends Amendment"
]

df = df.drop(columns=drop_cols)
df.columns

Index(['Legislation Number', 'URL', 'Congress', 'Title', 'Sponsor',
       'Date of Introduction', 'Committees', 'Latest Action',
       'Latest Action Date', 'Number of Cosponsors'],
      dtype='object')

In [76]:
# most frequent committees
df["Committees"].value_counts().head()

Committees
House - Energy and Commerce        277
House - Judiciary                  241
House - Ways and Means             219
House - Education and Workforce    153
House - Financial Services         152
Name: count, dtype: int64

In [77]:
# most frequent sponsors
df["Sponsor"].value_counts().head()

Sponsor
Lawler, Michael [Rep.-R-NY-17]                 33
Vindman, Eugene Simon [Rep.-D-VA-7]            29
Norton, Eleanor Holmes [Del.-D-DC-At Large]    25
Pappas, Chris [Rep.-D-NH-1]                    24
Gottheimer, Josh [Rep.-D-NJ-5]                 23
Name: count, dtype: int64

In [78]:
# most frequent latest action
df["Latest Action"].value_counts().head()

Latest Action
Referred to the House Committee on Energy and Commerce.        245
Referred to the House Committee on the Judiciary.              231
Referred to the House Committee on Ways and Means.             214
Referred to the House Committee on Education and Workforce.    138
Referred to the House Committee on Financial Services.         130
Name: count, dtype: int64

In [79]:
# parse sponsor name, party, and state

def parse_sponsor(s):
    match = re.match(r"^(.*?)\s+\[(?:Rep\.|Sen\.|Del\.)-([A-Z])-([A-Z]{2})-", str(s))
    if match:
        name, party_code, state = match.groups()
        party = "Democrat" if party_code == "D" else "Republican" if party_code == "R" else "Independent"
        # split "Last, First" → handle multi-word last names like "Watson Coleman, Bonnie"
        if "," in name:
            last, first = name.split(",", 1)
            last = last.strip()
            first = first.strip()
        else:
            last, first = name.strip(), None
        return first, last, party, state
    return None, s, None, None

df[["Sponsor First Name", "Sponsor Last Name", "Party", "State"]] = df["Sponsor"].apply(
    lambda x: pd.Series(parse_sponsor(x))
)

In [80]:
# drop sponsor col
df = df.drop(columns=["Sponsor"])

# make date of introduction and latest action date into datetime
df["Date of Introduction"] = pd.to_datetime(df["Date of Introduction"])
df["Latest Action Date"] = pd.to_datetime(df["Latest Action Date"])

# strip "House - " or "Senate - " prefix from Committees
df["Committees"] = df["Committees"].str.replace(r"^(House|Senate)\s*-\s*", "", regex=True)

df.head()

,Legislation Number,URL,Congress,Title,Date of Introduction,Committees,Latest Action,Latest Action Date,Number of Cosponsors,Sponsor First Name,Sponsor Last Name,Party,State
0,H.R. 7933,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),"To amend title 10, United States Code, to expa...",2026-03-12,Armed Services,Referred to the House Committee on Armed Servi...,2026-03-12,0,Bonnie,Watson Coleman,Democrat,NJ
1,H.R. 7932,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),HONOR Gold Star Families Act,2026-03-12,Armed Services,Referred to the House Committee on Armed Servi...,2026-03-12,11,Matt,Van Epps,Republican,TN
2,H.R. 7931,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),To amend the Employee Retirement Income Securi...,2026-03-12,Education and Workforce,Referred to the House Committee on Education a...,2026-03-12,1,Jefferson,Van Drew,Republican,NJ
3,H.R. 7930,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),"To amend title 18, United States Code, to incr...",2026-03-12,Judiciary,Referred to the House Committee on the Judiciary.,2026-03-12,6,Rashida,Tlaib,Democrat,MI
4,H.R. 7929,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),To direct the Secretary of Transportation to i...,2026-03-12,"Science, Space, and Technology","Referred to the House Committee on Science, Sp...",2026-03-12,0,Dina,Titus,Democrat,NV


In [81]:
# save cleaned data to csv
df.to_csv("../data/bills/bills_cleaned.csv", index=False)